# 종목별 우선순위 모델 정리

> 작성일: 2026-06-08  
> 목적: Choi(통계/Prophet) vs SU(ML/DL) 비교 결과를 바탕으로 종목별 1·2순위 모델 확정  
> 후속 작업: 데이터 드리프트 감지 기반 모델 자동 전환 DAG 작성

---

## 평가 방법론 주의사항

| 구분 | 평가 방식 | 평가 기간 |
|------|---------|----------|
| **Choi** | 실행일 1달 전 기준 1회 학습 → 이후 전체 기간 다단계 예측 | 최근 1달 |
| **SU** | 평가 날짜마다 재학습 후 D+5 rolling 예측 | 최근 1달 |

- Choi MAPE는 미래 1달 통예측 시나리오에 가까워 더 보수적
- SU MAPE는 매 시점 실제 데이터를 참고하므로 실제보다 낙관적일 수 있음
- 수치가 비슷한 경우 Choi 모델에 가중치를 둠

---

## SU 모델 — 전처리 및 예측 방식

### 데이터 수집

| 항목 | 내용 |
|------|------|
| 주가 데이터 | pykrx — 일별 OHLCV, 2021-01-01 ~ 실행일 |
| 매크로 데이터 | FinanceDataReader — S&P500, NASDAQ(NDX), VIX, USD/KRW |
| 저장 형식 | Parquet (`data/model_v2/{ticker}.parquet`) |
| 대상 종목 | 10종목 (KOSPI 대형주) |

### 피처 엔지니어링

```python
# 기본 9개 피처 (BASE_FEAT) — 전 종목 공통
ret_1d    = log(close / close.shift(1))          # 1일 로그 수익률
ret_5d    = log(close / close.shift(5))          # 5일 로그 수익률
ret_20d   = log(close / close.shift(20))         # 20일 로그 수익률
vol_norm  = volume / volume.rolling(20).mean()   # 정규화 거래량
kospi_ret = log(KOSPI / KOSPI.shift(1))          # KOSPI 일간 수익률
sp500_ret = log(SP500 / SP500.shift(1))          # S&P500 일간 수익률
ndx_ret   = log(NDX   / NDX.shift(1))            # NASDAQ 일간 수익률
usdkrw_ret= log(USDKRW/ USDKRW.shift(1))        # USD/KRW 환율 수익률
vix_chg   = VIX.diff()                           # VIX 일간 변화량

# 레짐 3개 피처 (RC 포함 모델만) — MSAR(k=2, order=1, switching_variance=True)
regime_prob     = bull 국면 확률 (0~1)
regime_duration = 현재 국면 연속 지속 거래일 수
regime_change   = 국면 전환 여부 (0 or 1)
```

### 타겟 변수

```python
target = log(close.shift(-5) / close)   # D+5 로그 수익률
```

### 학습 방법론

| 항목 | 내용 |
|------|------|
| CV 방식 | 국면 기반 Walk-forward (bull/bear 방향별 폴드 구성) |
| 가중치 | 국면 간 decay (`dp`) × 국면 내 decay (`dd`) — 최근 데이터에 높은 가중치 |
| 튜닝 | Optuna (n_trials=100~300, n_jobs=4, patience=40) |
| 평가 지표 | Rank IC (Spearman), eval 기간: 2025-06-01 ~ 2026-06-02 (48포인트, 5일 간격) |

### 예측 범위

| 모델 유형 | 입력 | 출력 | 예측 범위 |
|----------|------|------|----------|
| sklearn 계열 (ExtraTrees, XGB, LGBM 등) | 최신 1행 (9~12개 피처) | D+5 로그 수익률 1개 | **D+5 단일 포인트** |
| PatchTST | 512 거래일 × 9 피처 시퀀스 | D+1~D+5 로그 수익률 5개 → 합산 | **D+5 단일 포인트** |

> **SU 모델은 D+5 한 포인트만 예측**. 1달치 예측이 필요하면 매 거래일 rolling 추론 필요.

### PatchTST 아키텍처 요약

```
입력  (B, 512, 9)
  ↓ RevIN 정규화 (채널별 인스턴스 정규화)
  ↓ 패치 분할: (512-16)//8+1 = 63개 패치 × 16 크기
  ↓ Linear 임베딩 + 위치 인코딩 → (B×9, 63, 64)
  ↓ TransformerEncoder (2 레이어, nhead=4, ff=256, dropout=0.1)
  ↓ Linear Head: 63×64 → 5
출력  D+1~D+5 로그 수익률 → D+5 합산 사용
```

---

## Choi 모델 — 전처리 및 예측 방식

### 데이터 수집

| 항목 | 내용 |
|------|------|
| 주가 데이터 | yfinance (`{ticker}.KS`) — 2020-01-01 ~ 실행일 |
| 외생변수 | yfinance — USDKRW(`KRW=X`), WTI(`CL=F`), VIX(`^VIX`), KOSPI200(`^KS200`) |
| 전처리 탐색 | 8가지 × 학습 기간 8가지 = 64가지 조합 탐색 후 MAPE 최저 선택 |

### 전처리 옵션 (8가지)

| 옵션 | 변환 | 역변환 방식 |
|------|------|------------|
| `raw` | 원본 그대로 | 그대로 사용 |
| `log` | log(price) | exp(pred) |
| `diff1` | price.diff() | last_price + cumsum(pred) |
| `ret` | log(price).diff() | last_price × exp(cumsum(pred)) |
| `diff2` | price.diff().diff() | 2단계 cumsum |
| `log_diff2` | log(price).diff().diff() | 2단계 cumsum 후 exp |
| `seas5` | price.diff(5) | 5일 전 값 기준 rolling 복원 |
| `log_seas5` | log(price).diff(5) | 5일 전 log 기준 rolling 후 exp |

### 학습 기간 옵션 (8가지)

| 옵션 | 학습 시작 |
|------|----------|
| Super_Short | 실행일 - 6개월 |
| Short | 실행일 - 18개월 |
| Mid_Short | 실행일 - 24개월 |
| Recent | 실행일 - 30개월 |
| Mid | 실행일 - 36개월 |
| Mid_Long | 실행일 - 42개월 |
| Long | 실행일 - 48개월 |
| Full | 2020-01-01 고정 |

### 모델별 특성

| 모델 | 유형 | 입력 변수 | 특성 |
|------|------|----------|------|
| ARIMA(p,d,q) | 단변량 | 종가만 | 자기회귀 + 이동평균, 차분으로 정상성 확보 |
| ES (ExponentialSmoothing) | 단변량 | 종가만 | 추세 성분 포함, 지수평활 |
| VECM | 다변량 | 종가 + 거래량 + 외생변수 조합 | 공적분 관계 기반 오차수정 |
| Prophet | 단변량 | 종가만 | 연/주 seasonality, 변화점 자동 감지 |

### 예측 범위

| 항목 | 내용 |
|------|------|
| 학습 커트라인 | 실행일 기준으로 1회 고정 |
| 예측 방식 | 학습 후 `forecast(N)` 으로 N거래일 다단계 예측 |
| **예측 범위** | **실행일 이후 ~20 거래일 (약 1달) 전체** |
| 중간 데이터 참조 | 없음 — 예측 기간 중 실제 데이터 미사용 |

> **Choi 모델은 실행일 이후 1달치를 한 번에 예측** → DAG에서 1회 실행으로 전체 예측 생성 가능.

---

## 종목별 1순위 / 2순위 모델

| 종목 | 티커 | 1순위 모델 | 1순위 MAPE | 출처 | 예측 범위 | 2순위 모델 | 2순위 MAPE | 출처 | 예측 범위 |
|------|------|----------|----------|------|---------|----------|----------|------|---------|
| KB금융 | 105560 | ARIMA(3,0,0) | 1.56% | Choi | ~1달 전체 | LGBMRegressor(quantile) | 7.07% | SU | D+5 단일 |
| 신한지주 | 055550 | Prophet | 1.72% | Choi | ~1달 전체 | XGBRegressor | 2.08% | SU | D+5 단일 |
| 한화에어로스페이스 | 012450 | Prophet | 3.43% | Choi | ~1달 전체 | LGBMRegressor(quantile) | 11.94% | SU | D+5 단일 |
| 기아 | 000270 | Prophet | 3.52% | Choi | ~1달 전체 | ElasticNet | 7.44% | SU | D+5 단일 |
| LG화학 | 051910 | Prophet | 4.78% | Choi | ~1달 전체 | PatchTST | 8.08% | SU | D+5 단일 |
| SK이노베이션 | 096770 | LGBMRegressor(mse) | 5.21% | SU | D+5 단일 | ARIMA(0,0,3) | 5.36% | Choi | ~1달 전체 |
| LIG넥스원 | 079550 | HuberRegressor | 5.68% | SU | D+5 단일 | VECM | 5.70% | Choi | ~1달 전체 |
| 현대차 | 005380 | Prophet | 7.82% | Choi | ~1달 전체 | PatchTST | 9.87% | SU | D+5 단일 |
| 삼성전자 | 005930 | ExtraTreesRegressor | 5.09% | SU | D+5 단일 | Prophet | 10.22% | Choi | ~1달 전체 |
| SK하이닉스 | 000660 | PatchTST | 10.84% | SU | D+5 단일 | Prophet | 13.13% | Choi | ~1달 전체 |

In [ ]:
# DAG에서 import할 종목별 우선순위 모델 설정
# prediction_range:
#   'monthly' = 실행일 이후 ~20 거래일 한 번에 예측 (Choi 방식)
#   'd5'       = D+5 단일 포인트 예측, rolling 필요 (SU 방식)

MODEL_PRIORITY = {
    '105560': {
        'name': 'KB금융',
        'priority_1': {
            'model': 'ARIMA',
            'source': 'Choi',
            'mape': 1.56,
            'prediction_range': 'monthly',
            'config': {
                'order': (3, 0, 0),
                'preprocess': 'log',
                'train_window': 'Super_Short',  # 실행일 기준 -6개월
                'data_source': 'yfinance',
            }
        },
        'priority_2': {
            'model': 'LGBMRegressor',
            'source': 'SU',
            'mape': 7.07,
            'prediction_range': 'd5',
            'config': {
                'objective': 'quantile',
                'features': 9,  # BASE_FEAT (RC 미사용)
                'pkl': 'su/data/saved_models/105560.pkl',
                'data_source': 'pykrx+fdr',
            }
        }
    },
    '055550': {
        'name': '신한지주',
        'priority_1': {
            'model': 'Prophet',
            'source': 'Choi',
            'mape': 1.72,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'diff1',
                'train_window': 'Short',  # 실행일 기준 -18개월
                'seasonality': {'yearly': True, 'weekly': True, 'daily': False},
                'data_source': 'yfinance',
            }
        },
        'priority_2': {
            'model': 'XGBRegressor',
            'source': 'SU',
            'mape': 2.08,
            'prediction_range': 'd5',
            'config': {
                'features': 12,  # BASE_FEAT + regime 3개
                'pkl': 'su/data/saved_models/055550.pkl',
                'data_source': 'pykrx+fdr',
            }
        }
    },
    '012450': {
        'name': '한화에어로스페이스',
        'priority_1': {
            'model': 'Prophet',
            'source': 'Choi',
            'mape': 3.43,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'ret',       # log(price).diff()
                'train_window': 'Mid_Short',  # 실행일 기준 -24개월
                'seasonality': {'yearly': True, 'weekly': True, 'daily': False},
                'data_source': 'yfinance',
            }
        },
        'priority_2': {
            'model': 'LGBMRegressor',
            'source': 'SU',
            'mape': 11.94,
            'prediction_range': 'd5',
            'config': {
                'objective': 'quantile',
                'features': 12,  # BASE_FEAT + regime 3개
                'pkl': 'su/data/saved_models/012450.pkl',
                'data_source': 'pykrx+fdr',
            }
        }
    },
    '000270': {
        'name': '기아',
        'priority_1': {
            'model': 'Prophet',
            'source': 'Choi',
            'mape': 3.52,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'raw',
                'train_window': 'Recent',  # 실행일 기준 -30개월
                'seasonality': {'yearly': True, 'weekly': True, 'daily': False},
                'data_source': 'yfinance',
            }
        },
        'priority_2': {
            'model': 'ElasticNet',
            'source': 'SU',
            'mape': 7.44,
            'prediction_range': 'd5',
            'config': {
                'features': 12,  # BASE_FEAT + regime 3개, direction='bear'
                'pkl': 'su/data/saved_models/000270.pkl',
                'data_source': 'pykrx+fdr',
            }
        }
    },
    '051910': {
        'name': 'LG화학',
        'priority_1': {
            'model': 'Prophet',
            'source': 'Choi',
            'mape': 4.78,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'diff1',
                'train_window': 'Full',  # 2020-01-01 고정
                'seasonality': {'yearly': True, 'weekly': True, 'daily': False},
                'data_source': 'yfinance',
            }
        },
        'priority_2': {
            'model': 'PatchTST',
            'source': 'SU',
            'mape': 8.08,
            'prediction_range': 'd5',
            'config': {
                'features': 9,  # BASE_FEAT, seq_len=512
                'pkl': 'su/model/patchtst_v18_model.pkl',
                'state_dict_key': 'LG Chem',
                'data_source': 'pykrx+fdr',
            }
        }
    },
    '096770': {
        'name': 'SK이노베이션',
        'priority_1': {
            'model': 'LGBMRegressor',
            'source': 'SU',
            'mape': 5.21,
            'prediction_range': 'd5',
            'config': {
                'objective': 'mse',
                'features': 11,  # BASE_FEAT + regime_prob + regime_duration (RC 부분)
                'pkl': 'su/data/saved_models/096770.pkl',
                'data_source': 'pykrx+fdr',
            }
        },
        'priority_2': {
            'model': 'ARIMA',
            'source': 'Choi',
            'mape': 5.36,
            'prediction_range': 'monthly',
            'config': {
                'order': (0, 0, 3),
                'preprocess': 'raw',
                'train_window': 'Super_Short',  # 실행일 기준 -6개월
                'data_source': 'yfinance',
            }
        }
    },
    '079550': {
        'name': 'LIG넥스원',
        'priority_1': {
            'model': 'HuberRegressor',
            'source': 'SU',
            'mape': 5.68,
            'prediction_range': 'd5',
            'config': {
                'features': 12,  # BASE_FEAT + regime 3개, direction='bear'
                'pkl': 'su/data/saved_models/079550.pkl',
                'data_source': 'pykrx+fdr',
            }
        },
        'priority_2': {
            'model': 'VECM',
            'source': 'Choi',
            'mape': 5.70,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'level',
                'train_window': 'Mid',  # 실행일 기준 -36개월
                'exog_cols': ['KOSPI200', 'WTI', 'VIX'],
                'fixed_cols': ['close', 'volume'],
                'deterministic': 'co',
                'data_source': 'yfinance',
            }
        }
    },
    '005380': {
        'name': '현대차',
        'priority_1': {
            'model': 'Prophet',
            'source': 'Choi',
            'mape': 7.82,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'log',
                'train_window': 'Short',  # 실행일 기준 -18개월
                'seasonality': {'yearly': True, 'weekly': True, 'daily': False},
                'data_source': 'yfinance',
            }
        },
        'priority_2': {
            'model': 'PatchTST',
            'source': 'SU',
            'mape': 9.87,
            'prediction_range': 'd5',
            'config': {
                'features': 9,  # BASE_FEAT, seq_len=512
                'pkl': 'su/model/patchtst_v18_model.pkl',
                'state_dict_key': 'Hyundai Motor',
                'data_source': 'pykrx+fdr',
            }
        }
    },
    '005930': {
        'name': '삼성전자',
        'priority_1': {
            'model': 'ExtraTreesRegressor',
            'source': 'SU',
            'mape': 5.09,
            'prediction_range': 'd5',
            'config': {
                'features': 12,  # BASE_FEAT + regime 3개, direction='both'
                'pkl': 'su/data/saved_models/005930.pkl',
                'data_source': 'pykrx+fdr',
            }
        },
        'priority_2': {
            'model': 'Prophet',
            'source': 'Choi',
            'mape': 10.22,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'diff1',
                'train_window': 'Super_Short',  # 실행일 기준 -6개월
                'seasonality': {'yearly': True, 'weekly': True, 'daily': False},
                'data_source': 'yfinance',
            }
        }
    },
    '000660': {
        'name': 'SK하이닉스',
        'priority_1': {
            'model': 'PatchTST',
            'source': 'SU',
            'mape': 10.84,
            'prediction_range': 'd5',
            'config': {
                'features': 9,  # BASE_FEAT, seq_len=512
                'pkl': 'su/model/patchtst_v18_model.pkl',
                'state_dict_key': 'SK Hynix',
                'data_source': 'pykrx+fdr',
            }
        },
        'priority_2': {
            'model': 'Prophet',
            'source': 'Choi',
            'mape': 13.13,
            'prediction_range': 'monthly',
            'config': {
                'preprocess': 'log_diff2',
                'train_window': 'Recent',  # 실행일 기준 -30개월
                'seasonality': {'yearly': True, 'weekly': True, 'daily': False},
                'data_source': 'yfinance',
            }
        }
    },
}

print('MODEL_PRIORITY 로드 완료')
print()
for ticker, info in MODEL_PRIORITY.items():
    p1 = info['priority_1']
    p2 = info['priority_2']
    print(
        f"[{info['name']}({ticker})]"
        f"\n  1순위: {p1['model']:30s} MAPE={p1['mape']:5.2f}%  "
        f"범위={p1['prediction_range']:8s}  출처={p1['source']}"
        f"\n  2순위: {p2['model']:30s} MAPE={p2['mape']:5.2f}%  "
        f"범위={p2['prediction_range']:8s}  출처={p2['source']}"
        f"\n"
    )

---

## 데이터 드리프트 대응 전략

### 드리프트 감지 → 모델 전환 흐름

```
[매일 장 마감 후 실행]
  ↓
1. 최근 N일 실제 주가 수집
  ↓
2. 1순위 모델로 예측 → rolling MAPE 계산
  ↓
3. MAPE > threshold? ─── YES ──→ 2순위 모델로 전환
  │                                     ↓
  NO                          2순위도 MAPE > threshold?
  ↓                                     ↓ YES
계속 1순위 사용              → Slack 알림 + 재학습 DAG 트리거
```

### DAG 설계 포인트

| 항목 | 내용 |
|------|------|
| **실행 주기** | 매일 장 마감 후 (15:30 KST) |
| **드리프트 지표** | rolling 20일 MAPE |
| **전환 임계값** | 1순위 baseline MAPE × 1.5 |
| **재학습 트리거** | 2순위도 임계값 초과 시 재학습 DAG 호출 |
| **결과 저장** | S3 `predictions/{ticker}/{date}.json` |
| **prediction_range 주의** | 1·2순위 간 `monthly` ↔ `d5` 전환 시 출력 형식 다름 — DAG에서 통일 필요 |

In [ ]:
import pandas as pd

DRIFT_THRESHOLD_MULTIPLIER = 1.5

DRIFT_THRESHOLDS = {
    ticker: {
        'name': info['name'],
        'baseline_mape': info['priority_1']['mape'],
        'threshold': round(info['priority_1']['mape'] * DRIFT_THRESHOLD_MULTIPLIER, 2)
    }
    for ticker, info in MODEL_PRIORITY.items()
}

df_threshold = pd.DataFrame(DRIFT_THRESHOLDS).T[['name', 'baseline_mape', 'threshold']]
df_threshold.columns = ['종목명', '1순위 MAPE(%)', '전환 임계값(%)']
df_threshold = df_threshold.sort_values('1순위 MAPE(%)')
display(df_threshold)

---

## 다음 단계: DAG 작성 계획

```
airflow/dags/
├── finance_stock_predict_daily.py     ← 매일 예측 실행 + 드리프트 감지
└── finance_model_retrain_trigger.py   ← 드리프트 감지 시 재학습
```

**`finance_stock_predict_daily.py` 태스크 구성**
```
fetch_stock_data          (pykrx + yfinance + FDR)
  ↓
run_priority1_prediction  (10종목 병렬)
  ├─ Choi 1순위 (6종목): Prophet/ARIMA/VECM → monthly 예측 생성
  └─ SU 1순위   (4종목): pkl 로드 → D+5 단일 예측
  ↓
evaluate_drift            (rolling 20일 MAPE 계산)
  ↓
switch_to_priority2       (임계값 초과 종목만 2순위로 재실행)
  ↓
save_predictions_to_s3    (s3://fisa-news-archive/predictions/{ticker}/{date}.json)
  ↓
notify_if_retrain_needed  (2순위도 초과 시 알림)
```

**prediction_range 통일 방안**
- `monthly` 모델: 예측 결과에서 D+5 인덱스만 추출해 단일 포인트로 통일
- `d5` 모델: D+5 단일 포인트 그대로 사용
- 전체 monthly 예측 결과는 별도 컬럼으로 S3에 함께 저장